In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV , KFold , cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.base import BaseEstimator , TransformerMixin
import optuna
from lightgbm import LGBMRegressor
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice

In [2]:
dagshub.init(repo_owner="mridul0010" , repo_name="food-delivery-time-prediction" , mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/food-delivery-time-prediction"

Repository mridul0010/food-delivery-time-prediction initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow")

In [4]:
mlflow.set_experiment("3. Hyperparameter Tuning - RF")

2026/07/05 16:23:14 INFO mlflow.tracking.fluent: Experiment with name '3. Hyperparameter Tuning - RF' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/4e0f39779d504c019a0c189197867755', creation_time=1783248795597, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1783248795597, lifecycle_stage='active', name='3. Hyperparameter Tuning - RF', tags={}, trace_location=None, workspace='default'>

In [5]:
df = pd.read_csv('../data/Cleaned Delivery Dataset.csv')

In [6]:
pd.set_option('display.max_columns' , None)

In [7]:
df.drop(columns=['Delivery_Agent'] , inplace=True)

In [8]:
df['Order_Datetime'] = pd.to_datetime(df['Order_Datetime'])
df['Pickup_Datetime'] = pd.to_datetime(df['Pickup_Datetime'])

In [9]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.rating_bins = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # Store bins for consistent transformation
        self.rating_bins = pd.qcut(
            X['Delivery_person_Ratings'],
            q=3,
            retbins=True,
            duplicates='drop'
        )[1]
        
        return self

    def transform(self, X):
        X = X.copy()

        # Datetime conversion
        X['Order_Datetime'] = pd.to_datetime(X['Order_Datetime'], errors='coerce')
        X['Pickup_Datetime'] = pd.to_datetime(X['Pickup_Datetime'], errors='coerce')

        # Rating group
        X["delivery_rating_group"] = pd.cut(
            X['Delivery_person_Ratings'],
            bins=self.rating_bins,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

        # Age group
        X["age_group"] = pd.cut(
            X['Delivery_person_Age'],
            bins=[14, 25, 35, 60],
            labels=['Young', 'Adult', 'Senior']
        )

        # Distance group
        X["distance_group"] = pd.cut(
            X['distance_km'],
            bins=[0, 5, 10, 25],
            labels=['Short Distance', 'Medium Distance', 'Long Distance']
        )

        # Time features
        X['Prep_Time(min)'] = (
            X['Pickup_Datetime'] - X['Order_Datetime']
        ).dt.total_seconds() / 60

        X['Order_hour'] = X['Order_Datetime'].dt.hour
        X['Order_day'] = X['Order_Datetime'].dt.day_name()

        X['isWeekend'] = X['Order_day'].isin(["Saturday", "Sunday"]).astype(int)

        X['Time_Of_Day'] = pd.cut(
            X['Order_hour'],
            bins=[0, 6, 12, 18, 24],
            labels=["Night", "Morning", "Afternoon", "Evening"],
            include_lowest=True
        )

        # Drop raw datetime
        X = X.drop(columns=['Order_Datetime', 'Pickup_Datetime'], errors='ignore')

        return X

    def get_feature_names_out(self , input_features = None):
        return input_features

In [10]:
X = df.drop(columns=['Time_taken (min)'])
y = df['Time_taken (min)']

In [11]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [12]:
X

,City,Zone,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km,Order_Datetime,Pickup_Datetime,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City_Type
0,Dehradun,Zone17,36,4.2,30.327968,78.046106,30.397968,78.116106,10.28,2022-12-02 21:55:00,2022-12-02 22:10:00,Fog,Jam,Good,Snack,motorcycle,3,No,Metropolitian
1,Kochi,Zone16,21,4.7,10.003064,76.307589,10.043064,76.347589,6.24,2022-02-13 14:55:00,2022-02-13 15:05:00,Stormy,High,Average,Meal,motorcycle,1,No,Metropolitian
2,Pune,Zone13,23,4.7,18.562450,73.916619,18.652450,74.006619,13.79,2022-04-03 17:30:00,2022-04-03 17:40:00,Sandstorms,Medium,Average,Drinks,scooter,1,No,Metropolitian
3,Ludhiana,Zone15,34,4.3,30.899584,75.809346,30.919584,75.829346,2.93,2022-02-13 09:20:00,2022-02-13 09:30:00,Sandstorms,Low,poor,Buffet,motorcycle,0,No,Metropolitian
4,Kanpur,Zone14,24,4.7,26.463504,80.372929,26.593504,80.502929,19.40,2022-02-14 19:50:00,2022-02-14 20:05:00,Fog,Jam,Average,Snack,scooter,1,No,Metropolitian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39637,Bangalore,Zone16,28,4.9,13.029198,77.570997,13.059198,77.600997,4.66,2022-03-30 21:55:00,2022-03-30 22:05:00,Sandstorms,Jam,Average,Meal,scooter,1,No,Metropolitian
39638,Ranchi,Zone16,35,4.2,23.371292,85.327872,23.481292,85.437872,16.60,2022-08-03 21:45:00,2022-08-03 21:55:00,Windy,Jam,Good,Drinks,motorcycle,1,No,Metropolitian
39639,Chennai,Zone08,30,4.9,13.022394,80.242439,13.052394,80.272439,4.66,2022-11-03 23:50:00,2022-11-04 00:05:00,Cloudy,Low,Average,Drinks,scooter,0,No,Metropolitian
39640,Coimbatore,Zone11,20,4.7,11.001753,76.986241,11.041753,77.026241,6.23,2022-07-03 13:35:00,2022-07-03 13:40:00,Cloudy,High,poor,Snack,motorcycle,1,No,Metropolitian


In [13]:
numerical = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 
       'multiple_deliveries', 'distance_km', 'Prep_Time(min)', 'Order_hour',
       'isWeekend']

ohe_col = ['City', 'Zone', 'Weather_conditions',
       'Type_of_order', 'Type_of_vehicle',
       'City_Type','Order_day', 'Time_Of_Day']

ordinal_col = ['Road_traffic_density' , 'Vehicle_condition' , 'Festival',
       'delivery_rating_group' , 'age_group' , 'distance_group']

In [14]:
Road_traffic_density = ['Low', 'Medium', 'High', 'Jam']
Vehicle_condition = ['poor', 'Average', 'Good' , 'Excellent']
Festival = ['No', 'Yes']
delivery_rating_group = ['Low', 'Medium', 'High']
age_group = ['Young', 'Adult', 'Senior']         
distance_group = ['Short Distance', 'Medium Distance', 'Long Distance']

In [15]:
transformer = ColumnTransformer(
    transformers=[
        ("ohe" , OneHotEncoder(sparse_output=False , handle_unknown='ignore') , ohe_col),
        ('oe' , OrdinalEncoder(categories=[
            Road_traffic_density , Vehicle_condition,
            Festival , delivery_rating_group,
            age_group , distance_group
        ]) , ordinal_col),
        ('Scaling' , StandardScaler() , numerical)
    ],remainder='passthrough'
)

In [16]:
pipeline = Pipeline(
    [
        ("FeatureEngineering" , FeatureEngineering()),
        ("ColumnTransformer" , transformer)
    ]
) 

In [17]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

In [18]:
pt = PowerTransformer()

In [19]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            # Number of trees in the forest
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            
            # Maximum depth of the tree
            "max_depth": trial.suggest_int("max_depth", 2, 32),
            
            # Minimum number of samples required to split an internal node
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            
            # Minimum number of samples required to be at a leaf node
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            
            # Number of features to consider when looking for the best split
            "max_features": trial.suggest_float("max_features", 0.1, 1.0),
            
            # Whether bootstrap samples are used when building trees
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
            
            "random_state": 42,
            "n_jobs": -1,
        }

        # log params
        mlflow.log_params(params)


        rf = RandomForestRegressor(**params)
        model = TransformedTargetRegressor(regressor=rf , transformer=pt)

        model.fit(X_train_trans , y_train)

        cv_score = cross_val_score(
            model ,
            X_train_trans , 
            y_train , 
            cv=10 , 
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )

        mean_score = -(cv_score.mean())

        # log avg cross val error
        mlflow.log_metric("cross_val_error" , mean_score)

        return mean_score

In [20]:
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective , n_trials=50 , n_jobs=-1 , show_progress_bar=True)

    # log the best params
    mlflow.log_params(study.best_params)

    # log best score
    mlflow.log_metric("best_score" , study.best_value)

    # training the lgbm on best param
    best_rf = RandomForestRegressor(**study.best_params)

    best_model = TransformedTargetRegressor(regressor=best_rf , transformer=pt)

    best_model.fit(X_train_trans , y_train)

    y_pred_train = best_model.predict(X_train_trans)
    y_pred_test = best_model.predict(X_test_trans)

    scores = cross_val_score(
        best_model,
        X_train_trans,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=5,n_jobs=-1
    )

    # logging metrics
    mlflow.log_metric("Training_error" ,mean_absolute_error(y_train ,y_pred_train))
    mlflow.log_metric("Test_error" ,mean_absolute_error(y_test ,y_pred_test))
    mlflow.log_metric("Training_r2" ,r2_score(y_train ,y_pred_train))
    mlflow.log_metric("Test_r2" ,r2_score(y_test ,y_pred_test))
    mlflow.log_metric("cross_val" , -scores.mean())

    # Generate the optuna plots
    fig_history = plot_optimization_history(study)
    fig_parallel = plot_parallel_coordinate(study)
    fig_importance = plot_param_importances(study)
    fig_slice = plot_slice(study)

    # Loginf plots
    mlflow.log_figure(fig_history, "optuna_plots/optimization_history.html")
    mlflow.log_figure(fig_importance, "optuna_plots/param_importances.html")
    mlflow.log_figure(fig_parallel, "optuna_plots/parallel_coordinate.html")
    mlflow.log_figure(fig_slice, "optuna_plots/plot_slice.html")
    
    # log the best model 
    mlflow.sklearn.log_model(best_rf , artifact_path="model_RF")

[I 2026-07-05 16:23:15,330] A new study created in memory with name: no-name-3450cc49-da9e-4ba3-8673-0a7e55646cb1
  0%|          | 0/50 [00:00<?, ?it/s]

🏃 View run bold-wolf-30 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/3/runs/086e12d6b2e943c7bedbd25d8bcdaa01
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/3


Best trial: 6. Best value: 3.6067:   2%|▏         | 1/50 [01:11<58:11, 71.25s/it]

[I 2026-07-05 16:24:27,210] Trial 6 finished with value: 3.606703111945367 and parameters: {'n_estimators': 154, 'max_depth': 7, 'min_samples_split': 16, 'min_samples_leaf': 18, 'max_features': 0.3556766665934924, 'bootstrap': False}. Best is trial 6 with value: 3.606703111945367.
🏃 View run caring-swan-228 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/3/runs/987a608e01d543fcb5b40508bf69c718
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/3


Best trial: 3. Best value: 3.13539:   4%|▍         | 2/50 [01:30<32:25, 40.53s/it]

...

[I 2026-07-05 16:52:46,770] Trial 48 finished with value: 3.0555846918841807 and parameters: {'n_estimators': 256, 'max_depth': 28, 'min_samples_split': 9, 'min_samples_leaf': 12, 'max_features': 0.42219339612690393, 'bootstrap': False}. Best is trial 38 with value: 3.0503467002445985.
🏃 View run spiffy-lark-735 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/3/runs/40da8fbe1ba145aaaac12ecbae3fb3d7
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/3


Best trial: 38. Best value: 3.05035: 100%|██████████| 50/50 [30:12<00:00, 36.24s/it]


[I 2026-07-05 16:53:28,156] Trial 49 finished with value: 3.075094962973542 and parameters: {'n_estimators': 252, 'max_depth': 28, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': 0.5520342782566912, 'bootstrap': False}. Best is trial 38 with value: 3.0503467002445985.


2026/07/05 16:54:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/05 16:55:00 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.1+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.12.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


🏃 View run best_model at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/3/runs/ceea48a6eadf4eeeb02760bc66757384
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/3


In [21]:
study.best_value

3.0503467002445985

In [22]:
study.best_params

{'n_estimators': 243,
 'max_depth': 31,
 'min_samples_split': 5,
 'min_samples_leaf': 10,
 'max_features': 0.3572862510877454,
 'bootstrap': False}

In [23]:
best_rf = RandomForestRegressor(**study.best_params)

model = TransformedTargetRegressor(regressor=best_rf , transformer=pt)

model.fit(X_train_trans , y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",RandomForestR...stimators=243)
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",PowerTransformer()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,87
regressor_ regressor_: objectFitted regressor.,RandomForestRegressor,RandomForestR...stimators=243)
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,PowerTransformer,PowerTransformer()
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",243
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",31


In [24]:
y_pred = model.predict(X_test_trans)

r2_score(y_test , y_pred)

0.8285720894894504

In [25]:
study.trials_dataframe().head()

,number,value,datetime_start,datetime_complete,duration,params_bootstrap,params_max_depth,params_max_features,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,3.051411,2026-07-05 16:23:15.966034,2026-07-05 16:27:57.184811,0 days 00:04:41.218777,False,21,0.394706,8,5,206,COMPLETE
1,1,3.074762,2026-07-05 16:23:15.966640,2026-07-05 16:27:37.534427,0 days 00:04:21.567787,False,28,0.449642,4,10,132,COMPLETE
2,2,3.057181,2026-07-05 16:23:15.967470,2026-07-05 16:31:18.922358,0 days 00:08:02.954888,False,14,0.273606,6,10,257,COMPLETE
3,3,3.135386,2026-07-05 16:23:15.968560,2026-07-05 16:24:46.241324,0 days 00:01:30.272764,False,24,0.699061,1,20,84,COMPLETE
4,4,3.076766,2026-07-05 16:23:15.969312,2026-07-05 16:26:43.842820,0 days 00:03:27.873508,True,11,0.550994,4,2,102,COMPLETE


In [26]:
plot_optimization_history(study)

In [27]:
plot_param_importances(study)

In [28]:
plot_slice(study)

In [29]:
plot_parallel_coordinate(study)